# Chapter 18 — Multi-Agent Consensus and Cross-Document Consistency

This notebook accompanies **Chapter 18** of *How to Test and Validate Agentic AI Systems*.

**Setup:** We ingest two versions of the Meridian Financial Technologies
model risk assessment into separate GMS stores:

- `ch18_doc_q1` — the Q1 2026 quarterly report (original)
- `ch18_doc_q2` — the Q2 2026 post-remediation report (amended)

The Q2 report contains **deliberate factual changes**: MDL-001's risk rating
was downgraded from High to Medium after the fairness remediation, MDL-005
deteriorated further and was escalated, and MDL-013 was newly onboarded.
The `CrossDocumentAnalyzer` should surface these as *contradictions* while
leaving unchanged facts in the *minor_variance* band.

**Training:** Both stores ingest at 500 epochs so residual convergence drift is
small and any inter-store tension reflects genuine factual divergence.

In [ ]:
# === Preamble (all notebooks) ===
import sys
import os
import warnings
warnings.filterwarnings("ignore")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Ingest Two Versions of the Report

We create two GMS stores from two different revisions of the Meridian
MRA report. Both are trained at 500 epochs so convergence drift is small
and any inter-store tension is attributable to genuine factual conflict.

In [ ]:
from importlib.resources import files
# Store Q1 -- the original quarterly report
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config_q1 = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch18_doc_q1",
    n_runs=4,
    ingest_epochs=500,
    device=str(DEVICE),
)
bench_q1 = DOEGMSBenchmark(config_q1)
store_q1 = bench_q1.ingest()
print(f"Store Q1: {store_q1.stats()['entities']} entities, "
      f"{store_q1.stats()['triples']} triples, "
      f"{store_q1.stats()['enm_entries']} ENM")

In [ ]:
# Store Q2 -- post-remediation amended report
config_q2 = DOEHarnessConfig(
    markdown_path="knowlytix/harness/testing/tests/model_risk_assessment_q2.md",
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch18_doc_q2",
    n_runs=4,
    ingest_epochs=500,
    device=str(DEVICE),
)
bench_q2 = DOEGMSBenchmark(config_q2)
store_q2 = bench_q2.ingest()
print(f"Store Q2: {store_q2.stats()['entities']} entities, "
      f"{store_q2.stats()['triples']} triples, "
      f"{store_q2.stats()['enm_entries']} ENM")

In [4]:
# Copy calibration artifacts from the Ch 4 ground-truth directory to
# ch18_doc_q1 so the ConsensusOracle and CrossDocumentAnalyzer can read:
#   - calibration.{json,pkl}              per-claim tension (legacy)
#   - cross_store_calibration.{json,pkl}  data-driven 3-class bands
#                                         (tau_minor_upper, tau_contra_lower)
#                                         + isotonic-calibrated
#                                         CalibratedClassifierCV
# The cross-store artifacts are trained by
# scripts/train_cross_store_calibration.py against a labeled
# perturbation manifest; see Appendix G for the calibration pipeline.
import shutil
src_cal = "gmsh_testing_ext/lab_stores/ch04_ground_truth"
dst_cal = "gmsh_testing_ext/lab_stores/ch18_doc_q1"
for fname in [
    "calibration.json", "calibration.pkl",
    "cross_store_calibration.json", "cross_store_calibration.pkl",
    "cross_store_calibration_metrics.json",
]:
    src = os.path.join(src_cal, fname)
    dst = os.path.join(dst_cal, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"  copied {fname}")
    else:
        print(f"  {fname} missing at {src}")

# Read and display the cross-store calibration header.
# Schema v2 stores the 1-D projection under `projection_thresholds`
# (scan at tail_missing=0, has_replacement=1); the underlying
# classifier is 3-D.  Fall back to the legacy `thresholds` key for
# v1 artifacts so older calibration bundles still load.
import json
xs_json_path = os.path.join(src_cal, "cross_store_calibration.json")
if os.path.isfile(xs_json_path):
    xs_header = json.loads(open(xs_json_path).read())
    xs_taus = xs_header.get("projection_thresholds") or xs_header.get("thresholds", {})
    print("\n=== Cross-store calibration (from ch04_ground_truth) ===")
    print(f"  tau_minor_upper  = {xs_taus['tau_minor_upper']:.4f}"
          f"  (UNCHANGED <-> MODIFIED crossing)")
    print(f"  tau_contra_lower = {xs_taus['tau_contra_lower']:.4f}"
          f"  (MODIFIED <-> CONTRADICTED crossing)")
    print(f"  classifier       : {xs_header.get('classifier_type', '?')}")
    print(f"  calibration      : {xs_header.get('calibration_method', '?')}")


  copied calibration.json
  copied calibration.pkl
  copied cross_store_calibration.json
  copied cross_store_calibration.pkl
  copied cross_store_calibration_metrics.json

=== Cross-store calibration (from ch04_ground_truth) ===
  tau_minor_upper  = 0.1220  (UNCHANGED <-> MODIFIED crossing)
  tau_contra_lower = 0.1220  (MODIFIED <-> CONTRADICTED crossing)
  classifier       : CalibratedClassifierCV(LogisticRegression, cv=prefit)
  calibration      : isotonic


## Agent Roles and Mock Agents

`AgentRole` defines a per-role participant in a multi-agent workflow.
The mock agents return stub outputs; with real LLM-backed agents, the
analyst and reviewer would produce differentiated natural-language
responses that `ClaimCompiler` then extracts into comparable triples.

In [ ]:
# Define agent roles
from knowlytix.harness.testing.multi_agent import (
    AgentRole, ConsensusOracle, MultiAgentGenerator,
)
from knowlytix.harness.fixtures import create_agent_for_role

role_analyst = AgentRole(
    role_id="analyst",
    system_prompt="You are a credit risk analyst. Focus on AUC, KS, and discrimination.",
    allowed_tools=["read_report", "compute_metrics"],
    store_path="gmsh_testing_ext/lab_stores/ch18_doc_q1",
)
role_reviewer = AgentRole(
    role_id="reviewer",
    system_prompt="You are a compliance reviewer. Focus on fairness and disparate impact.",
    allowed_tools=["read_report", "check_fairness"],
    store_path="gmsh_testing_ext/lab_stores/ch18_doc_q2",
)

analyst = create_agent_for_role("analyst", expertise="credit risk")
reviewer = create_agent_for_role("reviewer", expertise="compliance")

print("=== Agent Roles ===")
print(f"  {role_analyst.role_id:<10s} -> store: {role_analyst.store_path}")
print(f"  {role_reviewer.role_id:<10s} -> store: {role_reviewer.store_path}")

In [ ]:
from importlib.resources import files
# Both agents answer the same question (using the Q1 report as context)
ctx = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()
q = "What is the risk rating of MDL-001?"

ans_analyst = analyst.run(f"<report>{ctx}</report>\n\nAnswer: {q}")
ans_reviewer = reviewer.run(f"<report>{ctx}</report>\n\nAnswer: {q}")

print("=== Agent Outputs (same question, Q1 report) ===")
print(f"  Question: {q}")
print(f"  Analyst:  '{ans_analyst}'")
print(f"  Reviewer: '{ans_reviewer}'")
print(f"  Agree:    {ans_analyst == ans_reviewer}")

## ConsensusOracle: Inter-Agent Tension

The `ConsensusOracle` computes pairwise tension for a given entity
across its agents' stores. When the two stores come from conflicting
source documents, the tension signal isolates the entities where the
two documents disagree.

In [7]:
# Baseline: both roles sharing the same store -- trivially zero tension
oracle_same = ConsensusOracle(stores={"analyst": store_q1, "reviewer": store_q1})
result_same = oracle_same.evaluate_consensus(entity="MDL-001")
print("=== ConsensusOracle -- same store (baseline) ===")
print("  Entity:           MDL-001")
print(f"  Tension thr.:     {oracle_same.tension_threshold:.4f}")
print(f"  Calibration:      {oracle_same._calibration_source}")
print(f"  Mean tension:     {result_same.mean_tension:.4f}")
print(f"  Consensus:        {result_same.consensus_reached}")
print(f"  Confidence:       {result_same.consensus_confidence:.4f}")
print(f"  Pairwise tension: {result_same.inter_agent_tension}")

=== ConsensusOracle -- same store (baseline) ===
  Entity:           MDL-001
  Tension thr.:     1.1271
  Calibration:      calibration.json
  Mean tension:     0.0000
  Consensus:        True
  Confidence:       1.0000
  Pairwise tension: {'analyst:reviewer': 0.0}


In [8]:
# ConsensusOracle with the two conflicting stores.
# The calibration_dir points at the Ch 4 ground-truth store, which holds
# the cross-store calibration artifacts (3-class bands + calibrated
# classifier).  When cross-store thresholds are loaded, consensus is
# declared only when mean tension falls below tau_minor_upper; the
# per-claim tau is retained as a fallback.
oracle = ConsensusOracle(
    stores={"q1": store_q1, "q2": store_q2},
    calibration_dir=src_cal,
)

entities_to_test = [
    "MDL-001", "MDL-002", "MDL-003",
    "MDL-005", "MDL-012", "CreditScore-XGB",
]

print("=== ConsensusOracle -- Q1 vs Q2 stores ===")
print(f"  Per-claim tau_tension (legacy):  {oracle.tension_threshold:.4f}"
      f"  [{oracle._calibration_source}]")
xs = oracle._cross_store_thresholds
if xs:
    print(f"  Cross-store tau_minor_upper:     {xs['tau_minor_upper']:.4f}"
          f"  (UNCHANGED/MODIFIED crossing)")
    print(f"  Cross-store tau_contra_lower:    {xs['tau_contra_lower']:.4f}"
          f"  (MODIFIED/CONTRADICTED crossing)")
    print(f"  Cross-store calibration source:  {oracle._cross_calibration_source}")
    print(f"  Cross-store classifier:          "
          f"{type(oracle._cross_store_classifier).__name__}")
else:
    print("  Cross-store calibration:         none loaded (falling back to per-claim tau)")
print()
print(f"{'Entity':<22s} {'Tension':>10s} {'Consensus':>10s} "
      f"{'Label':<16s} {'P(correct)':>12s}")
print("-" * 76)
for ent in entities_to_test:
    r = oracle.evaluate_consensus(entity=ent)
    t_val = list(r.inter_agent_tension.values())[0] if r.inter_agent_tension else None
    t_str = f"{t_val:.4f}" if t_val is not None else "N/A"
    print(f"  {ent:<20s} {t_str:>10s} {str(r.consensus_reached):>10s} "
          f"{(r.consensus_label or '-'):<16s} {r.consensus_confidence:>12.4f}")


=== ConsensusOracle -- Q1 vs Q2 stores ===
  Per-claim tau_tension (legacy):  1.1271  [calibration.json]
  Cross-store tau_minor_upper:     0.1220  (UNCHANGED/MODIFIED crossing)
  Cross-store tau_contra_lower:    0.1220  (MODIFIED/CONTRADICTED crossing)
  Cross-store calibration source:  dir:gmsh_testing_ext/lab_stores/ch04_ground_truth
  Cross-store classifier:          CalibratedClassifierCV

Entity                    Tension  Consensus Label              P(correct)
----------------------------------------------------------------------------


  MDL-001                  0.0698       True minor_variance         1.0000


  MDL-002                  0.0948      False contradiction          0.5416


  MDL-003                  0.0812       True minor_variance         1.0000


  MDL-005                  0.1109      False contradiction          0.5817


  MDL-012                  0.1100      False contradiction          0.5389
  CreditScore-XGB          0.2413       True minor_variance         1.0000


## Manual Cross-Store Triple Scoring

Score the *same* triples across both stores using actual relation names.
Triples that were deliberately changed between Q1 and Q2 should exhibit
much larger geodesic differences than triples that are identical in
both documents.

In [9]:
# Score triples from the Q1 store against both stores
triples_q1 = list(store_q1.doc_graph.triples)
triples_q2 = set(store_q2.doc_graph.triples)
scored = []
for h, r, t in triples_q1:
    s1 = store_q1.score_triple(h, r, t)
    s2 = store_q2.score_triple(h, r, t)
    if s1 is not None and s2 is not None:
        diff = abs(float(s1) - float(s2))
        in_q2 = (h, r, t) in triples_q2
        scored.append((h, r, t, float(s1), float(s2), diff, in_q2))

scored.sort(key=lambda x: x[5], reverse=True)

print(f"=== Cross-Store Triple Scoring ({len(scored)} triples) ===")
print(f"{'Triple':<60s} {'Q1':>8s} {'Q2':>8s} {'|Diff|':>8s} {'inQ2':>5s}")
print("-" * 95)
for h, r, t, s1, s2, diff, in_q2 in scored[:15]:
    label = f"({h}, {r}, {t})"[:58]
    print(f"  {label:<58s} {s1:>8.4f} {s2:>8.4f} {diff:>8.4f} {str(in_q2):>5s}")

import numpy as np
diffs = np.array([s[5] for s in scored])
print("\n=== Geodesic Difference Statistics ===")
print(f"  Mean: {diffs.mean():.4f}")
print(f"  Std:  {diffs.std():.4f}")
print(f"  Min:  {diffs.min():.4f}")
print(f"  Max:  {diffs.max():.4f}")

=== Cross-Store Triple Scoring (622 triples) ===
Triple                                                             Q1       Q2   |Diff|  inQ2
-----------------------------------------------------------------------------------------------
  (ACT-002, has_status, In Progress)                           0.9517   1.6653   0.7135 False
  (ACT-001, has_status, In Progress)                           1.0156   1.6954   0.6798 False
  (High-Risk Models, has_detail, 4)                            0.9440   1.5740   0.6299 False
  (ACT-003, has_status, Planned)                               1.0021   1.6012   0.5991 False
  (ACT-004, has_status, In Progress)                           1.0160   1.6132   0.5972 False
  (MDL-005, has_regulatory_penalty_risk, $0)                   0.8119   1.3984   0.5865 False
  (MDL-005, has_annual_loss_exposure, $0)                      0.8118   1.3765   0.5647 False
  (MDL-005, has_status, Degraded)                              1.0944   1.6312   0.5368 False
  (MDL-00

In [10]:
# Cross-store tension energy for entity pairs
print("=== Cross-Store Tension Energy ===")
entity_pairs = [
    ("MDL-001", "MDL-002"), ("MDL-001", "MDL-003"),
    ("MDL-002", "FraudNet-LSTM"),
    ("CreditScore-XGB", "Credit Decisioning"),
    ("High", "Medium"), ("MDL-005", "MDL-006"),
]
print(f"{'Pair':<45s} {'Q1':>10s} {'Q2':>10s} {'|Diff|':>10s}")
print("-" * 75)
for a, b in entity_pairs:
    try:
        te1 = store_q1.tension_energy(a, b)
        te2 = store_q2.tension_energy(a, b)
    except Exception:
        te1 = te2 = None
    if te1 is not None and te2 is not None:
        d = abs(float(te1) - float(te2))
        label = f"({a}, {b})"[:44]
        print(f"  {label:<43s} {float(te1):>10.4f} {float(te2):>10.4f} {d:>10.4f}")

=== Cross-Store Tension Energy ===
Pair                                                  Q1         Q2     |Diff|
---------------------------------------------------------------------------
  (MDL-001, MDL-002)                              0.7778     0.5497     0.2281
  (MDL-001, MDL-003)                              0.6895     0.5793     0.1102
  (MDL-002, FraudNet-LSTM)                        1.3859     1.4113     0.0255
  (CreditScore-XGB, Credit Decisioning)           1.3738     1.3329     0.0408
  (High, Medium)                                  1.3289     1.4008     0.0719
  (MDL-005, MDL-006)                              0.6369     0.5397     0.0972


## Delegation Holonomy

Holonomy defects quantify whether a delegation chain composes into a
direct relation. Small inter-store differences mean the relational
*structure* is stable across document revisions even when specific
values change.

In [11]:
# Delegation holonomy comparison
print("=== Delegation Holonomy ===")
test_chains = [
    (["has_risk_rating", "has_status"], "has_domain"),
    (["has_algorithm", "has_features"], "has_model_name"),
    (["has_tier", "has_risk_rating"], "has_status"),
]

print(f"{'Chain':<55s} {'Q1':>10s} {'Q2':>10s} {'|Diff|':>10s}")
print("-" * 90)
for chain, direct in test_chains:
    h1 = oracle.evaluate_delegation_holonomy(chain, direct, store_q1)
    h2 = oracle.evaluate_delegation_holonomy(chain, direct, store_q2)
    if h1 is not None and h2 is not None:
        label = f"{chain} -> {direct}"[:54]
        print(f"  {label:<54s} {h1:>10.4f} {h2:>10.4f} {abs(h1-h2):>10.4f}")

=== Delegation Holonomy ===
Chain                                                           Q1         Q2     |Diff|
------------------------------------------------------------------------------------------
  ['has_risk_rating', 'has_status'] -> has_domain            2.7181     2.6998     0.0183
  ['has_algorithm', 'has_features'] -> has_model_name        2.6521     2.6478     0.0043
  ['has_tier', 'has_risk_rating'] -> has_status              2.7083     2.7382     0.0299


## Lab 18.1 -- Multi-Agent Consensus Testing

Generate multi-agent coordination questions and examine the
`InteractionResult` audit artifact.

In [12]:
# Lab 18.1a -- Generate multi-agent questions
gen = MultiAgentGenerator()
questions = gen.generate(store_q1)

print("=== MultiAgentGenerator ===")
print(f"Generated {len(questions)} multi-agent questions\n")
print(f"{'QID':<20s} {'Entity':<20s} {'Type':<25s}")
print("-" * 68)
for q in questions[:8]:
    ent = q.metadata.get("entity", "?")
    ttype = q.metadata.get("test_type", q.category)
    print(f"  {q.qid:<18s} {ent:<20s} {ttype:<25s}")

=== MultiAgentGenerator ===
Generated 10 multi-agent questions

QID                  Entity               Type                     
--------------------------------------------------------------------
  multi_agent_000    0.7389               multi_agent_consistency  
  multi_agent_001    61.0                 multi_agent_consistency  
  multi_agent_002    82.0                 multi_agent_consistency  
  multi_agent_003    MDL-001              multi_agent_consistency  
  multi_agent_004    Critical             multi_agent_consistency  
  multi_agent_005    0.5123               multi_agent_consistency  
  multi_agent_006    Total Models in Inventory multi_agent_consistency  
  multi_agent_007    2026-04-15           multi_agent_consistency  


In [13]:
# Lab 18.1b -- Run both agents on multi-agent questions
print(f"{'QID':<20s} {'Analyst':<15s} {'Reviewer':<15s} {'Agree':>6s} {'GMS':>6s}")
print("-" * 68)
n_agree = 0
for q in questions[:10]:
    ans_a = analyst.run(q.natural_language)
    ans_r = reviewer.run(q.natural_language)
    agree = ans_a == ans_r
    gms_ok = q.gms_answer_fn(store_q1) if q.gms_answer_fn else None
    if agree:
        n_agree += 1
    a_short = (ans_a or "")[:13]
    r_short = (ans_r or "")[:13]
    print(f"  {q.qid:<18s} '{a_short}'   '{r_short}'   {str(agree):>6s} {str(gms_ok):>6s}")
print(f"\nAgreement rate: {n_agree}/{min(10, len(questions))} "
      f"({100.0*n_agree/max(1, min(10, len(questions))):.1f}%)")

QID                  Analyst         Reviewer         Agree    GMS
--------------------------------------------------------------------
  multi_agent_000    '0.8934'   '0.8934'     True   True
  multi_agent_001    '61.0'   '61.0'     True   True
  multi_agent_002    '0.8934'   '0.8934'     True   True
  multi_agent_003    '{'name': 'Cre'   '{'name': 'Cre'     True   True


  multi_agent_004    'No relevant m'   'No relevant m'     True   True


  multi_agent_005    '0.5123'   '0.5123'     True   True
  multi_agent_006    '| Total Model'   '| Total Model'     True   True
  multi_agent_007    'No relevant m'   'No relevant m'     True   True
  multi_agent_008    '{'name': 'Chu'   '{'name': 'Chu'     True   True
  multi_agent_009    '0.8934'   '0.8934'     True   True

Agreement rate: 10/10 (100.0%)


In [14]:
# Lab 18.1c -- InteractionResult from a live oracle run on MDL-001
result = oracle.evaluate_consensus(entity="MDL-001")
print("=== InteractionResult (live) ===")
for k, v in result.to_dict().items():
    print(f"  {k}: {v}")

=== InteractionResult (live) ===
  scenario_id: 
  n_agents: 0
  inter_agent_tension: {'q1:q2': 0.0698}
  process_holonomy: None
  consensus_reached: True
  mean_tension: 0.0698
  consensus_confidence: 1.0
  tau_tension: 1.1271
  consensus_label: minor_variance
  tau_minor_upper: 0.122
  tau_contra_lower: 0.122


## Lab 18.2 -- Cross-Document Conflict Detection

`CrossDocumentAnalyzer` scans shared entities across the Q1 and Q2
stores and classifies each disagreement into `minor_variance`,
`disagreement`, or `contradiction`. Bands are anchored on the calibrated
tau_tension and the tension ceiling (no hard-coded constants).

In [ ]:
# Lab 18.2a -- CrossDocumentAnalyzer: find shared entities.
# Pass ``calibration_dir=src_cal`` so the analyzer loads the data-driven
# 3-class cross-store bands instead of falling back to the single-tau
# per-claim regime.
from knowlytix.harness.testing.cross_document import (
    CrossDocumentAnalyzer, CrossDocumentGenerator, DocumentConflict,
    _conflict_type,
)

analyzer = CrossDocumentAnalyzer(
    stores={"q1": store_q1, "q2": store_q2},
    calibration_dir=src_cal,
)
shared = analyzer.find_shared_entities()

q1_entities = set(store_q1.adapter.entity_to_idx.keys())
q2_entities = set(store_q2.adapter.entity_to_idx.keys())
only_q1 = q1_entities - q2_entities
only_q2 = q2_entities - q1_entities

print("=== Shared Entities ===")
print(f"  Total Q1 entities: {len(q1_entities)}")
print(f"  Total Q2 entities: {len(q2_entities)}")
print(f"  Shared (both):     {len(shared)}")
print(f"  Only in Q1:        {len(only_q1)}  (first 5: {sorted(only_q1)[:5]})")
print(f"  Only in Q2:        {len(only_q2)}  (first 5: {sorted(only_q2)[:5]})")
print()
print(f"  Per-claim tau_tension (legacy):  {analyzer.tension_threshold:.4f}"
      f"  [{analyzer._calibration_source}]")
xs = analyzer._cross_store_thresholds
if xs:
    print(f"  Cross-store tau_minor_upper:     {xs['tau_minor_upper']:.4f}")
    print(f"  Cross-store tau_contra_lower:    {xs['tau_contra_lower']:.4f}")
    print(f"  Cross-store emit threshold:      {analyzer._emit_threshold:.4f}"
          f"  (= tau_minor_upper)")
    print(f"  Cross-store calibration source:  {analyzer._cross_calibration_source}")
    print(f"  Cross-store classifier:          "
          f"{type(analyzer._cross_store_classifier).__name__}")
else:
    print("  Cross-store calibration:         none loaded")
    print(f"  Per-claim tension classifier:    "
          f"{type(analyzer._tension_classifier).__name__}")


In [16]:
# Lab 18.2b -- Manual conflict detection across shared triples.
# We walk every Q1 triple, score it in both stores, and classify via the
# data-driven cross-store bands (tau_minor_upper / tau_contra_lower).
# The emit filter uses analyzer._emit_threshold = tau_minor_upper so that
# genuine-UNCHANGED triples are excluded by construction, not by an
# arbitrary magic constant.

tau_minor = analyzer._cross_store_thresholds.get("tau_minor_upper")
tau_contra = analyzer._cross_store_thresholds.get("tau_contra_lower")
emit = analyzer._emit_threshold

manual_conflicts = []
for h, r, t in store_q1.doc_graph.triples:
    s1 = store_q1.score_triple(h, r, t)
    s2 = store_q2.score_triple(h, r, t)
    if s1 is None or s2 is None:
        continue
    diff = abs(float(s1) - float(s2))
    if diff < emit:
        continue
    ctype = _conflict_type(
        diff,
        analyzer.tension_threshold,
        tau_minor_upper=tau_minor,
        tau_contra_lower=tau_contra,
    )
    conf = analyzer._conflict_confidence(diff, ctype)
    manual_conflicts.append(DocumentConflict(
        entity=h, store_a_id="q1", store_b_id="q2",
        fact_a=f"{h} {r} {t}", fact_b="(scored in q2)",
        inter_store_tension=diff,
        conflict_type=ctype,
        confidence=conf,
    ))
manual_conflicts.sort(key=lambda c: c.inter_store_tension, reverse=True)

print(f"=== Cross-Store Conflicts (delta >= {emit:.4f}, "
      f"data-driven 3-class bands) ===")
print(f"  Found {len(manual_conflicts)} conflicts above the emit threshold.")
if tau_minor is not None and tau_contra is not None:
    print(f"  Band boundaries: minor_variance <= {tau_minor:.4f} < "
          f"disagreement <= {tau_contra:.4f} < contradiction")
print()
print(f"{'Entity':<18s} {'Relation':<22s} {'Tail':<22s} "
      f"{'Tension':>8s} {'Type':<15s} {'Conf':>6s}")
print("-" * 100)
for c in manual_conflicts[:15]:
    parts = c.fact_a.split(' ', 2)
    h = parts[0] if parts else ""
    r = parts[1] if len(parts) > 1 else ""
    t = parts[2][:20] if len(parts) > 2 else ""
    print(f"  {h:<16s} {r:<22s} {t:<22s} {c.inter_store_tension:>8.4f} "
          f"{c.conflict_type:<15s} {c.confidence:>6.3f}")

print()
print("=== Conflict Type Distribution ===")
from collections import Counter
dist = Counter(c.conflict_type for c in manual_conflicts)
for k, v in dist.most_common():
    print(f"  {k:<18s}: {v}")


=== Cross-Store Conflicts (delta >= 0.1220, data-driven 3-class bands) ===
  Found 204 conflicts above the emit threshold.
  Band boundaries: minor_variance <= 0.1220 < disagreement <= 0.1220 < contradiction

Entity             Relation               Tail                    Tension Type              Conf
----------------------------------------------------------------------------------------------------
  ACT-002          has_status             In Progress              0.7135 contradiction    0.595
  ACT-001          has_status             In Progress              0.6798 contradiction    0.598
  High-Risk        Models                 has_detail 4             0.6299 contradiction    0.596
  ACT-003          has_status             Planned                  0.5991 contradiction    0.586
  ACT-004          has_status             In Progress              0.5972 contradiction    0.585
  MDL-005          has_regulatory_penalty_risk $0                       0.5865 contradiction    0.582
  MDL-

In [17]:
# Lab 18.2c -- Threshold sensitivity: count conflicts vs. alternative cuts.
# At the calibrated bands (tau_minor_upper, tau_contra_lower) the emitted
# conflicts are the cross-store "MODIFIED" + "CONTRADICTED" populations.
# Operators who retune these cuts should observe the expected monotone
# migration of the three band counts.

print("=== Conflict Count by Threshold ===")
if tau_minor is not None and tau_contra is not None:
    print(f"  Calibrated bands: tau_minor_upper={tau_minor:.4f}, "
          f"tau_contra_lower={tau_contra:.4f}\n")
else:
    print(f"  No cross-store bands loaded; falling back to "
          f"tau_tension={analyzer.tension_threshold:.4f}\n")

print(f"{'tau_minor':>10s} {'tau_contra':>11s} {'conflicts':>10s} "
      f"{'contradictions':>15s} {'disagreements':>14s} {'minor':>6s}")
print("-" * 70)

# Sweep tau_minor (emit threshold) while holding tau_contra at its
# calibrated value.  This is the knob a reviewer actually touches.
contra = tau_contra if tau_contra is not None else 0.6
for tm in [0.10, 0.15, tau_minor or 0.20, 0.25, 0.30, 0.35]:
    # Reclassify under (tm, contra); re-score Q1 triples
    surfaced = [c for c in manual_conflicts if c.inter_store_tension >= tm]
    reclassified = [
        _conflict_type(
            c.inter_store_tension,
            analyzer.tension_threshold,
            tau_minor_upper=tm,
            tau_contra_lower=contra,
        ) for c in surfaced
    ]
    n_contra = sum(x == "contradiction" for x in reclassified)
    n_disagr = sum(x == "disagreement" for x in reclassified)
    n_minor  = sum(x == "minor_variance" for x in reclassified)
    tag = " *" if tau_minor is not None and abs(tm - tau_minor) < 1e-6 else "  "
    print(f"  {tm:>8.4f}{tag} {contra:>11.4f} {len(surfaced):>10d} "
          f"{n_contra:>15d} {n_disagr:>14d} {n_minor:>6d}")
print()
print("  * = calibrated value (data-driven from labeled perturbation manifest)")


=== Conflict Count by Threshold ===
  Calibrated bands: tau_minor_upper=0.1220, tau_contra_lower=0.1220

 tau_minor  tau_contra  conflicts  contradictions  disagreements  minor
----------------------------------------------------------------------
    0.1000        0.1220        204             204              0      0
    0.1500        0.1220        166               0              0    166
    0.1220 *      0.1220        204             204              0      0
    0.2500        0.1220         60               0              0     60
    0.3000        0.1220         42               0              0     42
    0.3500        0.1220         34               0              0     34

  * = calibrated value (data-driven from labeled perturbation manifest)


In [18]:
# Lab 18.2d -- CrossDocumentGenerator with both stores
gen_cross = CrossDocumentGenerator(stores={"q2": store_q2})
cross_qs = gen_cross.generate(store_q1)

print("=== CrossDocumentGenerator (Q1 vs Q2) ===")
print(f"Generated {len(cross_qs)} cross-document questions\n")
for q in cross_qs[:5]:
    tension = q.metadata.get('tension', 'N/A')
    tt = q.metadata.get('test_type', q.category)
    print(f"  {q.qid}")
    print(f"    entity : {q.metadata.get('entity')}")
    print(f"    tension: {tension}")
    print(f"    label  : {q.ground_truth}")

=== CrossDocumentGenerator (Q1 vs Q2) ===
Generated 10 cross-document questions

  cross_doc_000
    entity : ACT-002
    tension: 0.7135
    label  : contradiction
  cross_doc_001
    entity : ACT-001
    tension: 0.6798
    label  : contradiction
  cross_doc_002
    entity : High-Risk Models
    tension: 0.6299
    label  : contradiction
  cross_doc_003
    entity : ACT-003
    tension: 0.5991
    label  : contradiction
  cross_doc_004
    entity : ACT-004
    tension: 0.5972
    label  : contradiction


## End-to-End: Full Cross-Store Scan

Score every triple in the Q1 report's graph against both stores and
report the full distribution across tension bands.

In [19]:
# E2E -- score all Q1 triples against both stores and bin by the
# data-driven 3-class cross-store bands.
all_scored = []
for h, r, t in store_q1.doc_graph.triples:
    s1 = store_q1.score_triple(h, r, t)
    s2 = store_q2.score_triple(h, r, t)
    if s1 is None or s2 is None:
        continue
    all_scored.append({"h": h, "r": r, "t": t,
                       "q1": float(s1), "q2": float(s2),
                       "diff": abs(float(s1) - float(s2))})

diffs = np.array([s['diff'] for s in all_scored])
print("=== E2E Cross-Store Scoring ===")
print(f"  Q1 triples:              {len(store_q1.doc_graph.triples)}")
print(f"  Q2 triples:              {len(store_q2.doc_graph.triples)}")
print(f"  Scored in both stores:   {len(all_scored)}")
print()
print("=== Geodesic Difference Distribution ===")
print(f"  Mean: {diffs.mean():.4f}")
print(f"  Std:  {diffs.std():.4f}")
print(f"  Min:  {diffs.min():.4f}")
print(f"  Max:  {diffs.max():.4f}")
print()
labels = [
    _conflict_type(
        s['diff'],
        analyzer.tension_threshold,
        tau_minor_upper=tau_minor,
        tau_contra_lower=tau_contra,
    ) for s in all_scored
]
from collections import Counter
cnt = Counter(labels)
if tau_minor is not None and tau_contra is not None:
    print("=== Conflict Classification (cross-store bands) ===")
    print(f"  tau_minor_upper  = {tau_minor:.4f}")
    print(f"  tau_contra_lower = {tau_contra:.4f}")
else:
    print("=== Conflict Classification (per-claim fallback) ===")
    print(f"  tau_tension      = {analyzer.tension_threshold:.4f}")
print()
for k in ['minor_variance', 'disagreement', 'contradiction']:
    n = cnt.get(k, 0)
    pct = 100.0 * n / len(all_scored) if all_scored else 0
    print(f"  {k:<18s}: {n:>4d} ({pct:.1f}%)")


=== E2E Cross-Store Scoring ===
  Q1 triples:              807
  Q2 triples:              811
  Scored in both stores:   622

=== Geodesic Difference Distribution ===
  Mean: 0.1148
  Std:  0.1150
  Min:  0.0001
  Max:  0.7135

=== Conflict Classification (cross-store bands) ===
  tau_minor_upper  = 0.1220
  tau_contra_lower = 0.1220

  minor_variance    :  418 (67.2%)
  disagreement      :    0 (0.0%)
  contradiction     :  204 (32.8%)


In [20]:
# E2E: holonomy comparison for multiple delegation chains
chain_tests = [
    (["has_risk_rating", "has_status"], "has_domain"),
    (["has_algorithm", "has_features"], "has_model_name"),
    (["has_tier", "has_risk_rating"], "has_status"),
    (["has_domain", "has_algorithm"], "has_tier"),
    (["has_status", "has_tier"], "has_risk_rating"),
]

print("=== E2E Holonomy Comparison ===")
print(f"{'Chain -> Direct':<55s} {'Q1':>9s} {'Q2':>9s} {'|Diff|':>9s}")
print("-" * 85)
diffs_h = []
for chain, direct in chain_tests:
    h1 = oracle.evaluate_delegation_holonomy(chain, direct, store_q1)
    h2 = oracle.evaluate_delegation_holonomy(chain, direct, store_q2)
    if h1 is None or h2 is None:
        continue
    d = abs(h1 - h2)
    diffs_h.append(d)
    label = f"{chain} -> {direct}"[:54]
    print(f"  {label:<54s} {h1:>9.4f} {h2:>9.4f} {d:>9.4f}")
print()
print(f"  Mean holonomy difference: {np.mean(diffs_h):.4f}")
print(f"  Max holonomy difference:  {np.max(diffs_h):.4f}")

=== E2E Holonomy Comparison ===
Chain -> Direct                                                Q1        Q2    |Diff|
-------------------------------------------------------------------------------------
  ['has_risk_rating', 'has_status'] -> has_domain           2.7181    2.6998    0.0183
  ['has_algorithm', 'has_features'] -> has_model_name       2.6521    2.6478    0.0043
  ['has_tier', 'has_risk_rating'] -> has_status             2.7083    2.7382    0.0299
  ['has_domain', 'has_algorithm'] -> has_tier               2.6672    2.6715    0.0042
  ['has_status', 'has_tier'] -> has_risk_rating             2.6782    2.7249    0.0467

  Mean holonomy difference: 0.0207
  Max holonomy difference:  0.0467


In [21]:
# Summary statistics
print("=== Chapter 18 Summary ===")
print(f"  Store Q1 (original, 500 epochs): "
      f"{store_q1.stats()['entities']} entities, "
      f"{store_q1.stats()['triples']} triples")
print(f"  Store Q2 (amended,  500 epochs): "
      f"{store_q2.stats()['entities']} entities, "
      f"{store_q2.stats()['triples']} triples")
print(f"  Shared entities:                 {len(shared)}")
print(f"  Entities only in Q1:             {len(only_q1)}")
print(f"  Entities only in Q2:             {len(only_q2)}")
print()
print(f"  Per-claim tau_tension (legacy):  {analyzer.tension_threshold:.4f}"
      f"  [{analyzer._calibration_source}]")
if tau_minor is not None and tau_contra is not None:
    print(f"  Cross-store tau_minor_upper:     {tau_minor:.4f}")
    print(f"  Cross-store tau_contra_lower:    {tau_contra:.4f}")
    print(f"  Cross-store calibration source:  {analyzer._cross_calibration_source}")
    print(f"  Cross-store classifier:          "
          f"{type(analyzer._cross_store_classifier).__name__}")
print()
print(f"  Triples scored in both:          {len(all_scored)}")
print(f"  Conflicts >= emit threshold:     {len(manual_conflicts)}")
print(f"    contradiction:                 "
      f"{sum(c.conflict_type == 'contradiction' for c in manual_conflicts)}")
print(f"    disagreement:                  "
      f"{sum(c.conflict_type == 'disagreement' for c in manual_conflicts)}")
print(f"    minor_variance:                "
      f"{sum(c.conflict_type == 'minor_variance' for c in manual_conflicts)}")

# Surface the calibration-run metrics so the reader sees what ECE the
# bands were tuned under and can audit per-class support / P / R / F1.
import os
import json
metrics_path = os.path.join(src_cal, "cross_store_calibration_metrics.json")
if os.path.isfile(metrics_path):
    m = json.loads(open(metrics_path).read())
    print()
    print(f"=== Cross-Store Calibration Quality "
          f"(from {os.path.basename(metrics_path)}) ===")
    ece = m.get("expected_calibration_error")
    if ece is not None:
        print(f"  Expected Calibration Error (ECE):    {ece:.4f}")
    brier = m.get("brier_score_ovr", {})
    if isinstance(brier, dict) and brier:
        brier_mean = sum(brier.values()) / len(brier)
        print(f"  Brier score (mean over classes):     {brier_mean:.4f}")
    print(f"  Test-set size:                       {m.get('test_size', '?')}")
    per_class = m.get("per_class", {})
    for cls in ("UNCHANGED", "MODIFIED", "CONTRADICTED"):
        if cls in per_class:
            pc = per_class[cls]
            print(f"  {cls:<14s} n={pc.get('support', 0):>3d}  "
                  f"P={pc.get('precision', 0):.3f}  "
                  f"R={pc.get('recall', 0):.3f}  "
                  f"F1={pc.get('f1', 0):.3f}")


=== Chapter 18 Summary ===
  Store Q1 (original, 500 epochs): 508 entities, 807 triples
  Store Q2 (amended,  500 epochs): 510 entities, 811 triples
  Shared entities:                 379
  Entities only in Q1:             129
  Entities only in Q2:             131

  Per-claim tau_tension (legacy):  1.1271  [calibration.json]
  Cross-store tau_minor_upper:     0.1220
  Cross-store tau_contra_lower:    0.1220
  Cross-store calibration source:  dir:gmsh_testing_ext/lab_stores/ch04_ground_truth
  Cross-store classifier:          CalibratedClassifierCV

  Triples scored in both:          622
  Conflicts >= emit threshold:     204
    contradiction:                 204
    disagreement:                  0
    minor_variance:                0

=== Cross-Store Calibration Quality (from cross_store_calibration_metrics.json) ===
  Expected Calibration Error (ECE):    0.1035
  Brier score (mean over classes):     0.0620
  Test-set size:                       50
  UNCHANGED      n= 27  P=0.963  

## Exercises

1. **Exercise 18.1.** Train a third store `ch18_doc_q3` that introduces a
   new contradiction (e.g., downgrade MDL-002 from High to Medium). Add it
   to the `CrossDocumentAnalyzer` and examine how the three-way conflict
   set compares with the pairwise view.

2. **Exercise 18.2.** Build an `InvarianceTester`-style wrapper that
   re-runs the Q1 -> Q2 scan with a tighter calibrated tau (e.g., half
   the calibrated value). How does the contradiction set shift?

3. **Exercise 18.3.** Write a three-agent consensus script:
   analyst reads Q1, reviewer reads Q2, approver reads both. Use
   `InteractionResult` to record whether the approver's decision is
   geometrically consistent with both inputs under holonomy measurement.